In [1]:
import sys

import pandas as pd
import os

# Adiciona a pasta pai ao caminho de busca do Python
sys.path.append(os.path.abspath('..'))

pd.options.display.float_format = '{:.2f}'.format

# Carregar os arquivos de 2014
dre = pd.read_csv("../DRE_tratado/dfp_dre_2014_filtrado.csv",              sep=";", encoding="utf-8")
bpa = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2014_filtrado.csv",     sep=";", encoding="utf-8")
bpp = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2014_filtrado.csv",  sep=";", encoding="utf-8")

# Filtrar setor petróleo
dre_pet = dre[dre['setor'] == 'Petroleo']
bpa_pet = bpa[bpa['setor'] == 'Petroleo']
bpp_pet = bpp[bpp['setor'] == 'Petroleo']

print("Empresas na DRE:", dre_pet['empresa'].unique())
print("Empresas no BPA:", bpa_pet['empresa'].unique())
print("Empresas no BPP:", bpp_pet['empresa'].unique())

Empresas na DRE: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str
Empresas no BPA: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str
Empresas no BPP: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str


In [2]:
# Ver todas as contas das duas empresas
print("=== PETROBRAS — DRE ===")
pet = dre_pet[dre_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
print(pet[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== PRIO — DRE ===")
prio = dre_pet[dre_pet['empresa'] == 'PRIO S.A.']
print(prio[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

=== PETROBRAS — DRE ===
codigo_conta                                                 descricao_conta         valor
        3.01                          Receita de Venda de Bens e/ou Serviços  337260000.00
        3.02                           Custo dos Bens e/ou Serviços Vendidos -256823000.00
        3.03                                                 Resultado Bruto   80437000.00
        3.04                                  Despesas/Receitas Operacionais -102353000.00
     3.04.01                                             Despesas com Vendas  -15974000.00
     3.04.02                               Despesas Gerais e Administrativas  -11223000.00
     3.04.03                      Perdas pela Não Recuperabilidade de Ativos          0.00
     3.04.04                                    Outras Receitas Operacionais          0.00
     3.04.05                                    Outras Despesas Operacionais  -75607000.00
  3.04.05.01                                                     T

In [3]:
print("=== PETROBRAS — BPP ===")
pet_bpp = bpp_pet[bpp_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
print(pet_bpp[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== PRIO — BPP ===")
prio_bpp = bpp_pet[bpp_pet['empresa'] == 'PRIO S.A.']
print(prio_bpp[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== CAIXA — BPA ===")
print(bpa_pet[bpa_pet['codigo_conta'] == '1.01.01'][['empresa', 'codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

=== PETROBRAS — BPP ===
 codigo_conta                                              descricao_conta        valor
            2                                                Passivo Total 793375000.00
         2.01                                           Passivo Circulante  82659000.00
      2.01.01                            Obrigações Sociais e Trabalhistas   5489000.00
   2.01.01.01                                           Obrigações Sociais         0.00
   2.01.01.02                                      Obrigações Trabalhistas         0.00
      2.01.02                                                 Fornecedores  25924000.00
   2.01.02.01                                       Fornecedores Nacionais         0.00
   2.01.02.02                                    Fornecedores Estrangeiros         0.00
      2.01.03                                           Obrigações Fiscais    657000.00
   2.01.03.01                                  Obrigações Fiscais Federais    657000.00
2.01.03.

In [4]:
# ============================================================
# EXTRAÇÃO E CÁLCULO — PETRÓLEO 2014
# ============================================================

ll        = dre_pet[dre_pet['codigo_conta'] == '3.11.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})
receita   = dre_pet[dre_pet['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
res_bruto = dre_pet[dre_pet['codigo_conta'] == '3.03'][['empresa', 'valor']].rename(columns={'valor': 'resultado_bruto'})
ebit      = dre_pet[dre_pet['codigo_conta'] == '3.05'][['empresa', 'valor']].rename(columns={'valor': 'ebit'})
desp_fin  = dre_pet[dre_pet['codigo_conta'] == '3.06.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_financeiras'})
pl        = bpp_pet[bpp_pet['codigo_conta'] == '2.03'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
div_cp    = bpp_pet[bpp_pet['codigo_conta'] == '2.01.04'][['empresa', 'valor']].rename(columns={'valor': 'div_circulante'})
div_lp    = bpp_pet[bpp_pet['codigo_conta'] == '2.02.01'][['empresa', 'valor']].rename(columns={'valor': 'div_nao_circulante'})
caixa     = bpa_pet[bpa_pet['codigo_conta'] == '1.01.01'][['empresa', 'valor']].rename(columns={'valor': 'caixa'})

# Verificar cobertura
print("=== COBERTURA ===")
for nome, df in [('lucro_liquido', ll), ('receita', receita), ('resultado_bruto', res_bruto),
                 ('ebit', ebit), ('desp_financeiras', desp_fin), ('pl', pl),
                 ('div_cp', div_cp), ('div_lp', div_lp), ('caixa', caixa)]:
    faltando = set(dre_pet['empresa'].unique()) - set(df['empresa'].unique())
    if faltando:
        print(f"❌ {nome}: faltando {faltando}")
    else:
        print(f"✅ {nome}: ambas as empresas")

# Montar e calcular
kpis = ll.merge(receita,   on='empresa')
kpis = kpis.merge(res_bruto, on='empresa')
kpis = kpis.merge(ebit,      on='empresa')
kpis = kpis.merge(desp_fin,  on='empresa')
kpis = kpis.merge(pl,        on='empresa')
kpis = kpis.merge(div_cp,    on='empresa')
kpis = kpis.merge(div_lp,    on='empresa')
kpis = kpis.merge(caixa,     on='empresa')

kpis['roe']            = kpis['lucro_liquido'] / kpis['patrimonio_liquido']
kpis['margem_liquida'] = kpis['lucro_liquido'] / kpis['receita']
kpis['margem_bruta']   = kpis['resultado_bruto'] / kpis['receita']
kpis['divida_bruta']   = kpis['div_circulante'] + kpis['div_nao_circulante']
kpis['divida_liquida'] = kpis['divida_bruta'] - kpis['caixa']
kpis['ebitda']         = kpis.apply(
    lambda r: r['ebit'] if r['ebit'] > 0 else None, axis=1
)
kpis['margem_ebitda']  = kpis.apply(
    lambda r: r['ebitda'] / r['receita'] if r['ebitda'] is not None else None, axis=1
)
kpis['alavancagem']    = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] is not None and r['ebitda'] > 0 else None, axis=1
)
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None, axis=1
)

from empresas_selecionadas import empresas_selecionadas
kpis['setor'] = kpis['empresa'].map(lambda x: empresas_selecionadas[x][0]) # type: ignore
kpis['tipo']  = kpis['empresa'].map(lambda x: empresas_selecionadas[x][1]) # type: ignore

cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print()
print(kpis[cols].to_string(index=False))

=== COBERTURA ===
✅ lucro_liquido: ambas as empresas
✅ receita: ambas as empresas
✅ resultado_bruto: ambas as empresas
✅ ebit: ambas as empresas
✅ desp_financeiras: ambas as empresas
✅ pl: ambas as empresas
✅ div_cp: ambas as empresas
✅ div_lp: ambas as empresas
✅ caixa: ambas as empresas

                           empresa    tipo   roe  margem_liquida  margem_bruta  margem_ebitda  alavancagem  cobertura_juros
                         PRIO S.A. Privado -1.61           -2.06          0.04            NaN          NaN           -16.29
         ENAUTA PARTICIPAÇÕES S.A. Privado  0.07            0.33          0.53           0.20         1.35             2.92
PETROLEO BRASILEIRO S.A. PETROBRAS Estatal -0.07           -0.06          0.24            NaN          NaN            -2.37


In [5]:
# Buscar depreciação nas duas empresas
print("=== DEPRECIAÇÃO — PETROBRAS ===")
print(dre_pet[
    (dre_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS') &
    (dre_pet['descricao_conta'].str.contains('depreci|amortiz', case=False, na=False))
][['codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

print("\n=== DEPRECIAÇÃO — PRIO ===")
print(dre_pet[
    (dre_pet['empresa'] == 'PRIO S.A.') &
    (dre_pet['descricao_conta'].str.contains('depreci|amortiz', case=False, na=False))
][['codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

=== DEPRECIAÇÃO — PETROBRAS ===
Empty DataFrame
Columns: [codigo_conta, descricao_conta, valor]
Index: []

=== DEPRECIAÇÃO — PRIO ===
codigo_conta        descricao_conta     valor
  3.04.02.06 Despesa de Depreciação -10090.00


In [6]:
# Para Petróleo — EBITDA com depreciação da Prio quando disponível
depr = dre_pet[dre_pet['codigo_conta'] == '3.04.02.06'][['empresa', 'valor']].rename(columns={'valor': 'depreciacao'})
kpis = kpis.merge(depr, on='empresa', how='left')
kpis['depreciacao'] = kpis['depreciacao'].fillna(0)

# EBITDA = EBIT + Depreciação (abs)
kpis['ebitda'] = kpis['ebit'] + kpis['depreciacao'].abs()

# Só calcular métricas quando EBITDA > 0
kpis['margem_ebitda']   = kpis.apply(
    lambda r: r['ebitda'] / r['receita'] if r['ebitda'] > 0 else None, axis=1
)
kpis['alavancagem']     = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] > 0 else None, axis=1
)
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None, axis=1
)

cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(kpis[cols].to_string(index=False))

                           empresa    tipo   roe  margem_liquida  margem_bruta  margem_ebitda  alavancagem  cobertura_juros
                         PRIO S.A. Privado -1.61           -2.06          0.04            NaN          NaN           -16.29
         ENAUTA PARTICIPAÇÕES S.A. Privado  0.07            0.33          0.53           0.20         1.35             2.92
PETROLEO BRASILEIRO S.A. PETROBRAS Estatal -0.07           -0.06          0.24            NaN          NaN            -2.37


In [7]:
import pandas as pd

for ano in range(2014, 2024):
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    
    termos = ['BRAVA', 'PETRORECONCAVO', 'RECÔNCAVO', 'RECONCAVO', 'ENAUTA', '3R PETROLEUM']
    
    matches = []
    for termo in termos:
        found = dre[dre['DENOM_CIA'].str.contains(termo, case=False, na=False)]['DENOM_CIA'].unique()
        matches.extend(found)
    
    if matches:
        print(f"{ano}: {list(set(matches))}")
    else:
        print(f"{ano}: nenhuma encontrada")

2014: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2015: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2016: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2017: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2018: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2019: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.', '3R PETROLEUM Ã\x93LEO E GÃ\x81S S.A.']
2020: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.', 'BRAVA ENERGIA S.A.']
2021: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.', 'BRAVA ENERGIA S.A.']
2022: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.', 'BRAVA ENERGIA S.A.']
2023: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.', 'BRAVA ENERGIA S.A.']


In [8]:
for ano in [2021, 2022, 2023]:
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    matches = dre[dre['DENOM_CIA'].str.contains('PETRO|PETR', case=False, na=False)]['DENOM_CIA'].unique()
    print(f"\n{ano}:")
    for m in matches:
        print(f"  {m}")


2021:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.

2022:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.

2023:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.


In [9]:
import pandas as pd

# Verificar nomes corretos com encoding latin1
for ano in [2019, 2020, 2021]:
    dre = pd.read_csv(
        f"../DRE_bruto/dfp_dre_{ano}.csv",
        sep=";",
        encoding="latin1"  # encoding original dos arquivos CVM
    )
    
    # Buscar todas as empresas de petróleo/gás
    termos = ['BRAVA', 'ENAUTA', '3R PETROLEUM', 'PETRORECONCAVO', 'PETROREC']
    for termo in termos:
        found = dre[
            dre['DENOM_CIA'].str.contains(termo, case=False, na=False)
        ]['DENOM_CIA'].unique()
        if len(found) > 0:
            print(f"{ano} | {termo}: {list(found)}")

2019 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2019 | 3R PETROLEUM: ['3R PETROLEUM Ã\x93LEO E GÃ\x81S S.A.']
2020 | BRAVA: ['BRAVA ENERGIA S.A.']
2020 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2020 | PETROREC: ['PETRORECÃ\x94NCAVO S.A.']
2021 | BRAVA: ['BRAVA ENERGIA S.A.']
2021 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2021 | PETROREC: ['PETRORECÃ\x94NCAVO S.A.']


In [10]:
import pandas as pd

# Ler com encoding latin1 e decodificar corretamente
# para ver os nomes sem caracteres quebrados
dre_2021 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2021.csv",
    sep=";",
    encoding="latin1"  # encoding original CVM
)

# Filtrar empresas de petróleo e exibir nome limpo
termos = ['BRAVA', 'ENAUTA', '3R PETROLEUM', 'PETROREC']
for termo in termos:
    found = dre_2021[
        dre_2021['DENOM_CIA'].str.contains(termo, case=False, na=False)
    ]['DENOM_CIA'].unique()
    for nome in found:
        # Tentar decodificar o nome corretamente
        try:
            nome_limpo = nome.encode('latin1').decode('utf-8')
        except:
            nome_limpo = nome
        print(f"Original: {nome}")
        print(f"Limpo:    {nome_limpo}")
        print()

Original: BRAVA ENERGIA S.A.
Limpo:    BRAVA ENERGIA S.A.

Original: ENAUTA PARTICIPAÃÃES S.A.
Limpo:    ENAUTA PARTICIPAÇÕES S.A.

Original: PETRORECÃNCAVO S.A.
Limpo:    PETRORECÔNCAVO S.A.



In [11]:
# Confirmar nome correto do 3R Petroleum
dre_2019 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2019.csv",
    sep=";",
    encoding="latin1"  # encoding original CVM
)

# Buscar 3R e decodificar nome
found = dre_2019[
    dre_2019['DENOM_CIA'].str.contains('3R', case=False, na=False)
]['DENOM_CIA'].unique()

for nome in found:
    try:
        # Converter de latin1 para utf-8
        nome_limpo = nome.encode('latin1').decode('utf-8')
    except:
        nome_limpo = nome
    print(f"Original: {nome}")
    print(f"Limpo:    {nome_limpo}")

Original: 3R PETROLEUM ÃLEO E GÃS S.A.
Limpo:    3R PETROLEUM ÓLEO E GÁS S.A.


In [12]:
import pandas as pd

# Carregar DRE de 2014 já filtrada para verificar estrutura
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Filtrar BRB e Banestes
bancos = ['BRB BANCO DE BRASILIA S.A.', 'BANESTES S.A. - BCO EST ESPIRITO SANTO']

for banco in bancos:
    print(f"\n=== {banco} — CONTAS DRE 2014 ===")
    df = dre_2014[dre_2014['DENOM_CIA'] == banco]
    
    # Filtrar só exercício atual e versão mais recente
    df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']
    df = df.sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CD_CONTA'])
    
    # Mostrar contas relevantes para os KPIs
    contas_relevantes = ['3.01', '3.02', '3.03', '3.04.02', '3.04.03', 
                         '3.04.05', '3.09.01', '3.11.01']
    print(df[df['CD_CONTA'].isin(contas_relevantes)][
        ['CD_CONTA', 'DS_CONTA', 'VL_CONTA']
    ].sort_values('CD_CONTA').to_string(index=False))


=== BRB BANCO DE BRASILIA S.A. — CONTAS DRE 2014 ===
Empty DataFrame
Columns: [CD_CONTA, DS_CONTA, VL_CONTA]
Index: []

=== BANESTES S.A. - BCO EST ESPIRITO SANTO — CONTAS DRE 2014 ===
Empty DataFrame
Columns: [CD_CONTA, DS_CONTA, VL_CONTA]
Index: []


In [13]:
import pandas as pd

# Carregar DRE bruta de 2014
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Verificar como aparece o campo ORDEM_EXERC para o BRB
brb = dre_2014[dre_2014['DENOM_CIA'] == 'BRB BANCO DE BRASILIA S.A.']

print("=== VALORES ÚNICOS DE ORDEM_EXERC ===")
print(brb['ORDEM_EXERC'].unique())

print("\n=== PRIMEIRAS LINHAS DO BRB ===")
print(brb[['CD_CONTA', 'DS_CONTA', 'ORDEM_EXERC', 'VL_CONTA']].head(10).to_string(index=False))

=== VALORES ÚNICOS DE ORDEM_EXERC ===
<StringArray>
['PENÃLTIMO', 'ÃLTIMO']
Length: 2, dtype: str

=== PRIMEIRAS LINHAS DO BRB ===
CD_CONTA                                   DS_CONTA ORDEM_EXERC    VL_CONTA
    3.01     Receitas da IntermediaÃ§Ã£o Financeira  PENÃLTIMO  1910786.00
    3.01     Receitas da IntermediaÃ§Ã£o Financeira     ÃLTIMO  2181587.00
    3.02     Despesas da IntermediaÃ§Ã£o Financeira  PENÃLTIMO  -544224.00
    3.02     Despesas da IntermediaÃ§Ã£o Financeira     ÃLTIMO  -808074.00
    3.03 Resultado Bruto IntermediaÃ§Ã£o Financeira  PENÃLTIMO  1366562.00
    3.03 Resultado Bruto IntermediaÃ§Ã£o Financeira     ÃLTIMO  1373513.00
    3.04      Outras Despesas/Receitas Operacionais  PENÃLTIMO -1047714.00
    3.04      Outras Despesas/Receitas Operacionais     ÃLTIMO -1050881.00
 3.04.01       Receitas de PrestaÃ§Ã£o de ServiÃ§os  PENÃLTIMO   416469.00
 3.04.01       Receitas de PrestaÃ§Ã£o de ServiÃ§os     ÃLTIMO   484934.00


In [14]:
import pandas as pd

# Carregar DRE bruta de 2014
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Filtrar BRB e Banestes
bancos = ['BRB BANCO DE BRASILIA S.A.', 'BANESTES S.A. - BCO EST ESPIRITO SANTO']

for banco in bancos:
    print(f"\n=== {banco} — CONTAS DRE 2014 ===")
    df = dre_2014[dre_2014['DENOM_CIA'] == banco].copy()
    
    # Filtrar exercício atual — usar str.contains por causa do encoding quebrado
    df = df[df['ORDEM_EXERC'].str.contains('LTIMO') & 
            ~df['ORDEM_EXERC'].str.contains('PEN')]
    
    # Manter só versão mais recente
    df = df.sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CD_CONTA'])
    
    # Mostrar contas relevantes para os KPIs financeiros
    contas_relevantes = ['3.01', '3.02', '3.03', '3.04.01', '3.04.02', 
                         '3.04.03', '3.04.05', '3.09.01', '3.11.01']
    resultado = df[df['CD_CONTA'].isin(contas_relevantes)][
        ['CD_CONTA', 'DS_CONTA', 'VL_CONTA']
    ].sort_values('CD_CONTA')
    
    print(resultado.to_string(index=False))


=== BRB BANCO DE BRASILIA S.A. — CONTAS DRE 2014 ===
CD_CONTA                                     DS_CONTA   VL_CONTA
    3.01       Receitas da IntermediaÃ§Ã£o Financeira 2181587.00
    3.02       Despesas da IntermediaÃ§Ã£o Financeira -808074.00
    3.03   Resultado Bruto IntermediaÃ§Ã£o Financeira 1373513.00
 3.04.01         Receitas de PrestaÃ§Ã£o de ServiÃ§os  484934.00
 3.04.02                          Despesas de Pessoal -724605.00
 3.04.03              Outras Despesas Administrativas -422544.00
 3.04.05                 Outras Receitas Operacionais  147700.00
 3.09.01 AtribuÃ­do a SÃ³cios da Empresa Controladora  197462.00

=== BANESTES S.A. - BCO EST ESPIRITO SANTO — CONTAS DRE 2014 ===
CD_CONTA                                     DS_CONTA    VL_CONTA
    3.01       Receitas da IntermediaÃ§Ã£o Financeira  1752613.00
    3.02       Despesas da IntermediaÃ§Ã£o Financeira -1069151.00
    3.03   Resultado Bruto IntermediaÃ§Ã£o Financeira   683462.00
 3.04.01         Receitas de Pr

In [15]:
import pandas as pd

pd.options.display.float_format = '{:.2f}'.format

# Carregar arquivos de 2014 já tratados pelo ETL
dre = pd.read_csv("../DRE_tratado/dfp_dre_2014_filtrado.csv",              sep=";", encoding="utf-8", keep_default_na=False)
bpa = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2014_filtrado.csv",     sep=";", encoding="utf-8", keep_default_na=False)
bpp = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2014_filtrado.csv",  sep=";", encoding="utf-8", keep_default_na=False)

# Filtrar setor petróleo
dre_pet = dre[dre['setor'] == 'Petroleo']
bpa_pet = bpa[bpa['setor'] == 'Petroleo']
bpp_pet = bpp[bpp['setor'] == 'Petroleo']

# Confirmar quais empresas aparecem em 2014
print("=== EMPRESAS PETRÓLEO 2014 ===")
print("DRE:", dre_pet['empresa'].unique())
print("BPA:", bpa_pet['empresa'].unique())
print("BPP:", bpp_pet['empresa'].unique())

=== EMPRESAS PETRÓLEO 2014 ===
DRE: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str
BPA: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str
BPP: <StringArray>
['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.',
 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 3, dtype: str


In [16]:
# ============================================================
# EXTRAÇÃO DAS CONTAS — PETRÓLEO 2014
# Contas idênticas para todas as empresas do setor
# ============================================================

# DRE — Resultado
ll        = dre_pet[dre_pet['codigo_conta'] == '3.11.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})
receita   = dre_pet[dre_pet['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
res_bruto = dre_pet[dre_pet['codigo_conta'] == '3.03'][['empresa', 'valor']].rename(columns={'valor': 'resultado_bruto'})
ebit      = dre_pet[dre_pet['codigo_conta'] == '3.05'][['empresa', 'valor']].rename(columns={'valor': 'ebit'})
desp_fin  = dre_pet[dre_pet['codigo_conta'] == '3.06.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_financeiras'})

# BPP — Passivo e Patrimônio
pl        = bpp_pet[bpp_pet['codigo_conta'] == '2.03'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
div_cp    = bpp_pet[bpp_pet['codigo_conta'] == '2.01.04'][['empresa', 'valor']].rename(columns={'valor': 'div_circulante'})
div_lp    = bpp_pet[bpp_pet['codigo_conta'] == '2.02.01'][['empresa', 'valor']].rename(columns={'valor': 'div_nao_circulante'})

# BPA — Caixa
caixa     = bpa_pet[bpa_pet['codigo_conta'] == '1.01.01'][['empresa', 'valor']].rename(columns={'valor': 'caixa'})

# Depreciação — conta específica da Prio (3.04.02.06)
# Petrobras não reporta depreciação na DRE consolidada
depr      = dre_pet[dre_pet['codigo_conta'] == '3.04.02.06'][['empresa', 'valor']].rename(columns={'valor': 'depreciacao'})

# Verificar cobertura de todas as contas
print("=== COBERTURA DAS CONTAS ===")
todas_empresas = set(dre_pet['empresa'].unique())
for nome, df in [
    ('lucro_liquido',    ll),
    ('receita',          receita),
    ('resultado_bruto',  res_bruto),
    ('ebit',             ebit),
    ('desp_financeiras', desp_fin),
    ('pl',               pl),
    ('div_cp',           div_cp),
    ('div_lp',           div_lp),
    ('caixa',            caixa),
    ('depreciacao',      depr),
]:
    faltando = todas_empresas - set(df['empresa'].unique())
    if faltando:
        print(f"⚠️  {nome}: faltando {faltando}")
    else:
        print(f"✅ {nome}: todas as empresas")

=== COBERTURA DAS CONTAS ===
✅ lucro_liquido: todas as empresas
✅ receita: todas as empresas
✅ resultado_bruto: todas as empresas
✅ ebit: todas as empresas
✅ desp_financeiras: todas as empresas
✅ pl: todas as empresas
✅ div_cp: todas as empresas
✅ div_lp: todas as empresas
✅ caixa: todas as empresas
⚠️  depreciacao: faltando {'PETROLEO BRASILEIRO S.A. PETROBRAS', 'ENAUTA PARTICIPAÇÕES S.A.'}


In [17]:
# ============================================================
# MONTAGEM DO DATAFRAME E CÁLCULO DOS KPIs — PETRÓLEO 2014
# ============================================================

# Juntar todas as contas em um único dataframe
kpis = ll.merge(receita,   on='empresa')
kpis = kpis.merge(res_bruto, on='empresa')
kpis = kpis.merge(ebit,      on='empresa')
kpis = kpis.merge(desp_fin,  on='empresa')
kpis = kpis.merge(pl,        on='empresa')
kpis = kpis.merge(div_cp,    on='empresa')
kpis = kpis.merge(div_lp,    on='empresa')
kpis = kpis.merge(caixa,     on='empresa')

# Depreciação com how='left' pois nem todas as empresas têm
kpis = kpis.merge(depr, on='empresa', how='left')

# Preencher depreciação ausente com 0
kpis['depreciacao'] = kpis['depreciacao'].fillna(0)

# ============================================================
# CÁLCULO DOS KPIs
# ============================================================

# ROE — Retorno sobre Patrimônio Líquido
kpis['roe']            = kpis['lucro_liquido'] / kpis['patrimonio_liquido']

# Margem Líquida — eficiência na conversão de receita em lucro
kpis['margem_liquida'] = kpis['lucro_liquido'] / kpis['receita']

# Margem Bruta — eficiência antes das despesas operacionais
kpis['margem_bruta']   = kpis['resultado_bruto'] / kpis['receita']

# Dívida Bruta e Líquida
kpis['divida_bruta']   = kpis['div_circulante'] + kpis['div_nao_circulante']
kpis['divida_liquida'] = kpis['divida_bruta'] - kpis['caixa']

# EBITDA = EBIT + Depreciação (em módulo pois depreciação é despesa negativa)
kpis['ebitda']         = kpis['ebit'] + kpis['depreciacao'].abs()

# Margem EBITDA — só calcular quando EBITDA > 0
kpis['margem_ebitda']  = kpis.apply(
    lambda r: r['ebitda'] / r['receita'] if r['ebitda'] > 0 else None, axis=1
)

# Alavancagem — Dívida Líquida / EBITDA (só quando EBITDA > 0)
kpis['alavancagem']    = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] > 0 else None, axis=1
)

# Cobertura de Juros — EBIT / Despesas Financeiras
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None, axis=1
)

# Adicionar setor e tipo
from empresas_selecionadas import empresas_selecionadas, get_tipo, get_setor
kpis['setor'] = kpis['empresa'].map(lambda x: get_setor(x))
kpis['tipo']  = kpis['empresa'].map(lambda x: get_tipo(x, 2014))

# Exibir resultado
cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(kpis[cols].sort_values('roe', ascending=False).to_string(index=False))

                           empresa    tipo   roe  margem_liquida  margem_bruta  margem_ebitda  alavancagem  cobertura_juros
         ENAUTA PARTICIPAÇÕES S.A. Privado  0.07            0.33          0.53           0.20         1.35             2.92
PETROLEO BRASILEIRO S.A. PETROBRAS Estatal -0.07           -0.06          0.24            NaN          NaN            -2.37
                         PRIO S.A. Privado -1.61           -2.06          0.04            NaN          NaN           -16.29


In [18]:
# ============================================================
# VERIFICAR COBERTURA POR ANO — PETRÓLEO
# Quantas empresas privadas temos em cada ano?
# ============================================================

for ano in range(2014, 2024):
    # Carregar DRE tratada do ano
    dre_ano = pd.read_csv(
        f"../DRE_tratado/dfp_dre_{ano}_filtrado.csv",
        sep=";",
        encoding="utf-8",
        keep_default_na=False
    )
    
    # Filtrar setor petróleo
    pet = dre_ano[dre_ano['setor'] == 'Petroleo']
    
    # Contar estatais e privados
    from empresas_selecionadas import get_tipo
    empresas = pet['empresa'].unique()
    estatais = [e for e in empresas if get_tipo(e, ano) == 'Estatal']
    privados = [e for e in empresas if get_tipo(e, ano) == 'Privado']
    
    print(f"{ano}: {len(estatais)} estatais {estatais} | {len(privados)} privados {privados}")

2014: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 2 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2015: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 2 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2016: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 2 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2017: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 2 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2018: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 2 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2019: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 3 privados ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2020: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 4 privados ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2021: 1 estatais ['PETROLEO BRASILEIRO S.A. PETROBRAS'] | 4 privados ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA

In [19]:
# ============================================================
# VERIFICAR 3R PETROLEUM — por que sumiu após 2019?
# ============================================================

for ano in [2019, 2020, 2021]:
    dre_ano = pd.read_csv(
        f"../DRE_bruto/dfp_dre_{ano}.csv",
        sep=";",
        encoding="latin1"
    )
    
    # Buscar qualquer empresa com '3R' no nome
    matches = dre_ano[
        dre_ano['DENOM_CIA'].str.contains('3R', case=False, na=False)
    ]['DENOM_CIA'].unique()
    
    print(f"{ano}: {list(matches)}")

2019: ['3R PETROLEUM Ã\x93LEO E GÃ\x81S S.A.']
2020: []
2021: []


In [20]:
# ============================================================
# BUSCAR 3R PELO CÓDIGO CVM
# O código CVM não muda mesmo quando a empresa muda de nome
# ============================================================

# Primeiro pegar o código CVM do 3R em 2019
dre_2019 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2019.csv",
    sep=";",
    encoding="latin1"
)

# Buscar código CVM do 3R
codigo_3r = dre_2019[
    dre_2019['DENOM_CIA'].str.contains('3R', case=False, na=False)
]['CD_CVM'].unique()

print(f"Código CVM do 3R: {codigo_3r}")

# Agora buscar esse código nos anos seguintes
for ano in [2020, 2021, 2022, 2023]:
    dre_ano = pd.read_csv(
        f"../DRE_bruto/dfp_dre_{ano}.csv",
        sep=";",
        encoding="latin1"
    )
    
    # Buscar pelo código CVM
    for cod in codigo_3r:
        matches = dre_ano[dre_ano['CD_CVM'] == cod]['DENOM_CIA'].unique()
        if len(matches) > 0:
            print(f"{ano} (CVM {cod}): {list(matches)}")
        else:
            print(f"{ano} (CVM {cod}): não encontrado")

Código CVM do 3R: [25291]
2020 (CVM 25291): ['BRAVA ENERGIA S.A.']
2021 (CVM 25291): ['BRAVA ENERGIA S.A.']
2022 (CVM 25291): ['BRAVA ENERGIA S.A.']
2023 (CVM 25291): ['BRAVA ENERGIA S.A.']


In [21]:
import pandas as pd
from empresas_selecionadas import get_tipo

# ============================================================
# VERIFICAR COBERTURA DO PETRÓLEO POR ANO
# Quantas empresas estatais e privadas temos em cada ano?
# ============================================================

for ano in range(2014, 2024):
    # Carregar DRE tratada do ano
    dre_ano = pd.read_csv(
        f"../DRE_tratado/dfp_dre_{ano}_filtrado.csv",
        sep=";",
        encoding="utf-8",
        keep_default_na=False
    )
    
    # Filtrar setor petróleo
    pet = dre_ano[dre_ano['setor'] == 'Petroleo']
    empresas = pet['empresa'].unique()
    
    # Separar estatais e privados pelo tipo no ano
    estatais = [e for e in empresas if get_tipo(e, ano) == 'Estatal']
    privados = [e for e in empresas if get_tipo(e, ano) == 'Privado']
    
    print(f"{ano}: {len(estatais)} estatal | {len(privados)} privados → {list(privados)}")

2014: 1 estatal | 2 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2015: 1 estatal | 2 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2016: 1 estatal | 2 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2017: 1 estatal | 2 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2018: 1 estatal | 2 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
2019: 1 estatal | 3 privados → ['PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2020: 1 estatal | 4 privados → ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2021: 1 estatal | 4 privados → ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2022: 1 estatal | 4 privados → ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']
2023: 1 estatal | 4 privados → ['PETRORECÔNCAVO S.A.', 'PRIO S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'BRAVA ENERGIA S.A.']


In [22]:
import pandas as pd
import zipfile
import os

# ============================================================
# VERIFICAR SE DVA EXISTE NOS ARQUIVOS BRUTOS
# A DVA (Demonstração do Valor Adicionado) pode conter
# a depreciação que não aparece na DRE consolidada
# ============================================================

# Verificar estrutura do ZIP de 2014
import requests
from zipfile import ZipFile
from io import BytesIO

url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_2014.zip"

print("Baixando ZIP de 2014 para verificar arquivos disponíveis...")
r = requests.get(url, timeout=60)
z = ZipFile(BytesIO(r.content))

# Listar todos os arquivos dentro do ZIP
print("\nArquivos disponíveis no ZIP:")
for nome in sorted(z.namelist()):
    print(f"  {nome}")

Baixando ZIP de 2014 para verificar arquivos disponíveis...

Arquivos disponíveis no ZIP:
  dfp_cia_aberta_2014.csv
  dfp_cia_aberta_BPA_con_2014.csv
  dfp_cia_aberta_BPA_ind_2014.csv
  dfp_cia_aberta_BPP_con_2014.csv
  dfp_cia_aberta_BPP_ind_2014.csv
  dfp_cia_aberta_DFC_MD_con_2014.csv
  dfp_cia_aberta_DFC_MD_ind_2014.csv
  dfp_cia_aberta_DFC_MI_con_2014.csv
  dfp_cia_aberta_DFC_MI_ind_2014.csv
  dfp_cia_aberta_DMPL_con_2014.csv
  dfp_cia_aberta_DMPL_ind_2014.csv
  dfp_cia_aberta_DRA_con_2014.csv
  dfp_cia_aberta_DRA_ind_2014.csv
  dfp_cia_aberta_DRE_con_2014.csv
  dfp_cia_aberta_DRE_ind_2014.csv
  dfp_cia_aberta_DVA_con_2014.csv
  dfp_cia_aberta_DVA_ind_2014.csv
  dfp_cia_aberta_parecer_2014.csv


In [23]:
import pandas as pd
from io import BytesIO

# ============================================================
# LER DVA E BUSCAR DEPRECIAÇÃO DAS EMPRESAS DE PETRÓLEO
# A DVA decompõe o valor adicionado pela empresa, incluindo
# a depreciação como item explícito
# ============================================================

# Ler DVA consolidada de 2014
dva = pd.read_csv(
    z.open("dfp_cia_aberta_DVA_con_2014.csv"),
    sep=";",
    encoding="latin1"
)

print("=== COLUNAS DA DVA ===")
print(list(dva.columns))

print("\n=== AMOSTRA ===")
print(dva.head(3).to_string())

# Filtrar empresas de petróleo
empresas_pet = [
    'PETROLEO BRASILEIRO S.A. PETROBRAS',
    'PRIO S.A.',
    'ENAUTA PARTICIPAÇÕES S.A.'
]

# Buscar contas relacionadas a depreciação
print("\n=== DEPRECIAÇÃO NA DVA — PETROBRAS ===")
pet = dva[dva['DENOM_CIA'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
depr = pet[pet['DS_CONTA'].str.contains('depreci|amortiz', case=False, na=False)]
print(depr[['CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ORDEM_EXERC']].to_string(index=False))

print("\n=== DEPRECIAÇÃO NA DVA — ENAUTA ===")
# Buscar com encoding quebrado e limpo
enauta_names = ['ENAUTA PARTICIPAÃÃES S.A.', 'ENAUTA PARTICIPAÇÕES S.A.']
en = dva[dva['DENOM_CIA'].isin(enauta_names)]
depr_en = en[en['DS_CONTA'].str.contains('depreci|amortiz', case=False, na=False)]
print(depr_en[['CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ORDEM_EXERC']].to_string(index=False))

=== COLUNAS DA DVA ===
['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP', 'MOEDA', 'ESCALA_MOEDA', 'ORDEM_EXERC', 'DT_INI_EXERC', 'DT_FIM_EXERC', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ST_CONTA_FIXA']

=== AMOSTRA ===
             CNPJ_CIA    DT_REFER  VERSAO        DENOM_CIA  CD_CVM                                          GRUPO_DFP MOEDA ESCALA_MOEDA ORDEM_EXERC DT_INI_EXERC DT_FIM_EXERC CD_CONTA                  DS_CONTA     VL_CONTA ST_CONTA_FIXA
0  00.000.000/0001-91  2014-12-31       2  BCO BRASIL S.A.    1023  DF Consolidado - Demonstração de Valor Adicionado  REAL          MIL   PENÚLTIMO   2013-01-01   2013-12-31     7.01                  Receitas 114798690.00             S
1  00.000.000/0001-91  2014-12-31       2  BCO BRASIL S.A.    1023  DF Consolidado - Demonstração de Valor Adicionado  REAL          MIL      ÚLTIMO   2014-01-01   2014-12-31     7.01                  Receitas 146989548.00             S
2  00.000.000/0001-91  2014-12-31       2  BCO BRASIL

In [24]:
# ============================================================
# VERIFICAR DEPRECIAÇÃO NA DVA PARA TODAS AS EMPRESAS DE PETRÓLEO
# Conta 7.04.01 — Depreciação, Amortização e Exaustão
# ============================================================

# Filtrar só exercício atual
dva_atual = dva[
    dva['ORDEM_EXERC'].str.contains('LTIMO') &
    ~dva['ORDEM_EXERC'].str.contains('PEN')
]

print("=== DEPRECIAÇÃO (7.04.01) POR EMPRESA — 2014 ===")
depr_todas = dva_atual[dva_atual['CD_CONTA'] == '7.04.01']

# Buscar empresas de petróleo
empresas_pet = [
    'PETROLEO BRASILEIRO S.A. PETROBRAS',
    'PRIO S.A.',
    'ENAUTA PARTICIPAÃÃES S.A.',  # encoding quebrado
    'ENAUTA PARTICIPAÇÕES S.A.',  # encoding correto
]

resultado = depr_todas[depr_todas['DENOM_CIA'].isin(empresas_pet)][
    ['DENOM_CIA', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA']
]
print(resultado.to_string(index=False))

=== DEPRECIAÇÃO (7.04.01) POR EMPRESA — 2014 ===
                         DENOM_CIA CD_CONTA                            DS_CONTA     VL_CONTA
                         PRIO S.A.  7.04.01 Depreciação, Amortização e Exaustão   -176338.00
         ENAUTA PARTICIPAÇÕES S.A.  7.04.01 Depreciação, Amortização e Exaustão   -117613.00
PETROLEO BRASILEIRO S.A. PETROBRAS  7.04.01 Depreciação, Amortização e Exaustão -30677000.00


In [25]:
import requests
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

# ============================================================
# VERIFICAR DEPRECIAÇÃO NA DVA PARA TODOS OS ANOS
# Confirmar que conta 7.04.01 existe para as empresas
# de petróleo em todos os anos 2014-2023
# ============================================================

empresas_pet = [
    'PETROLEO BRASILEIRO S.A. PETROBRAS',
    'PRIO S.A.',
    'ENAUTA PARTICIPAÃÃES S.A.',   # encoding quebrado
    'ENAUTA PARTICIPAÇÕES S.A.',   # encoding correto
    'BRAVA ENERGIA S.A.',
    '3R PETROLEUM ÃLEO E GÃS S.A.',  # encoding quebrado
    '3R PETROLEUM ÓLEO E GÁS S.A.',  # encoding correto
    'PETRORECÃNCAVO S.A.',            # encoding quebrado
    'PETRORECÔNCAVO S.A.',            # encoding correto
]

print("=== COBERTURA DA DEPRECIAÇÃO (DVA 7.04.01) POR ANO ===\n")

for ano in range(2014, 2024):
    # Baixar ZIP do ano
    url = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
    r = requests.get(url, timeout=60)
    z = ZipFile(BytesIO(r.content))
    
    # Ler DVA consolidada
    dva = pd.read_csv(
        z.open(f"dfp_cia_aberta_DVA_con_{ano}.csv"),
        sep=";",
        encoding="latin1"
    )
    
    # Filtrar exercício atual e conta de depreciação
    dva_atual = dva[
        dva['ORDEM_EXERC'].str.contains('LTIMO') &
        ~dva['ORDEM_EXERC'].str.contains('PEN') &
        (dva['CD_CONTA'] == '7.04.01')
    ]
    
    # Filtrar empresas de petróleo
    encontradas = dva_atual[
        dva_atual['DENOM_CIA'].isin(empresas_pet)
    ]['DENOM_CIA'].unique()
    
    # Decodificar nomes
    nomes_limpos = []
    for nome in encontradas:
        try:
            nomes_limpos.append(nome.encode('latin1').decode('utf-8'))
        except:
            nomes_limpos.append(nome)
    
    print(f"{ano}: {sorted(set(nomes_limpos))}")

=== COBERTURA DA DEPRECIAÇÃO (DVA 7.04.01) POR ANO ===

2014: ['ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2015: ['ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2016: ['ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2017: ['ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2018: ['ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2019: ['3R PETROLEUM ÓLEO E GÁS S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PRIO S.A.']
2020: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PETRORECÔNCAVO S.A.', 'PRIO S.A.']
2021: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PETRORECÔNCAVO S.A.', 'PRIO S.A.']
2022: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÇÕES S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS', 'PETRORECÔNCAVO S.A.', 'PRIO S

In [26]:
import pandas as pd

# ============================================================
# VERIFICAR SE EBITDA E ALAVANCAGEM ESTÃO PREENCHIDOS
# Comparar antes (sem DVA) e depois (com DVA)
# ============================================================

kpis_pet = pd.read_csv(
    "../KPIs/kpis_petroleo.csv",
    sep=";",
    encoding="utf-8",
    keep_default_na=False
)

# Ver KPIs da Petrobras ao longo dos anos
print("=== PETROBRAS — KPIs POR ANO ===")
pet = kpis_pet[kpis_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
cols = ['ano', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(pet[cols].to_string(index=False)) 

print("\n=== NULOS POR COLUNA ===")
# Contar valores vazios (string vazia ou NaN)
for col in ['margem_ebitda', 'alavancagem', 'cobertura_juros']:
    vazios = kpis_pet[col].isin(['', 'None']).sum() + kpis_pet[col].isna().sum()
    print(f"{col}: {vazios} vazios de {len(kpis_pet)} linhas")

=== PETROBRAS — KPIs POR ANO ===
 ano    tipo   roe  margem_liquida  margem_bruta margem_ebitda alavancagem  cobertura_juros
2014 Estatal -0.07           -0.06          0.24         0.026     35.0184            -2.37
2015 Estatal -0.14           -0.11          0.31        0.0789     15.5599            -0.40
2016 Estatal -0.06           -0.05          0.32        0.2301      4.8701             0.53
2017 Estatal -0.00           -0.00          0.32        0.2829      3.5761             1.08
2018 Estatal  0.09            0.07          0.36        0.3102      2.5158             1.98
2019 Estatal  0.13            0.13          0.40        0.4795      2.2181             2.06
2020 Estatal  0.02            0.03          0.46        0.4152      2.9361             0.95
2021 Estatal  0.27            0.24          0.49        0.6199      0.9601             3.31
2022 Estatal  0.52            0.29          0.52         0.576       0.647            10.26
2023 Estatal  0.33            0.24          0.5

In [ ]:
import pandas as pd

# ============================================================
# VERIFICAR QUAIS LINHAS AINDA TÊM NULOS NO PETRÓLEO
# ============================================================

kpis_pet = pd.read_csv(
    "../KPIs/kpis_petroleo.csv",
    sep=";",
    encoding="utf-8",
    keep_default_na=False
)

# Encontrar linhas com margem_ebitda vazia
vazios = kpis_pet[
    (kpis_pet['margem_ebitda'] == '') | 
    (kpis_pet['margem_ebitda'].isna())
][['empresa', 'ano', 'tipo', 'roe', 'margem_ebitda', 'alavancagem']]

print("=== LINHAS COM MARGEM EBITDA VAZIA ===")
print(vazios.to_string(index=False))

=== LINHAS COM MARGEM EBITDA VAZIA ===
                  empresa  ano    tipo   roe margem_ebitda alavancagem
                PRIO S.A. 2014 Privado -1.61                          
ENAUTA PARTICIPAÇÕES S.A. 2015 Privado  0.03                          
       BRAVA ENERGIA S.A. 2019 Privado -0.14                          
       BRAVA ENERGIA S.A. 2020 Privado -0.24                          
